# STNet long memory test (stage 2 only)

In the main run the long-range LSTM trained for about 17 seconds, so "it added nothing" didn't tell us much. Here I take a finished stage-1 network, keep it frozen, and train only the LSTM for much longer. Then every head is scored again.

Two budgets, same LSTM width (64):

- lmb (hlmb): 8 epochs x 300 steps. Probably 30 to 45 minutes. I haven't timed it.
- lmc (hlmc): 30 epochs x 600 steps. Probably 3 to 4 hours. Not timed either. Time cap is 8 hours per job.

An epoch here means one pass over the 3 long-memory years. With two GPUs the two jobs run side by side. With one GPU they run one after the other, so allow about 4.5 to 5 hours in total (a guess).

This needs a CUDA GPU (T4 or P100). A TPU notebook can't run it.

## Which weights

- `['lmb', 'lmc']` starts from the main run (`stnet_s42_final.pt`, trained on all fault kinds).
- `['hlmb', 'hlmc']` starts from the hold run (`stnet_hold_s42_final.pt`), which never saw zeroed, scale or drift. The LSTM is trained without them too. This shows whether long memory helps on kinds it has never seen.

Set `RUN_JOBS` in the next cell. Only the `_final.pt` file is read.

## What counts as a result

The only real hypothesis is drift. A drift fault ramps up over 21 to 60 days, longer than the 16-day window the encoder sees. Scale and zeroed faults last 1 to 7 days, so they fit inside that window. I treat them as controls: they shouldn't move, and if they do the cause isn't memory.

Long memory helped if, against the stage-1 run on the same benchmark, drift gains at least 0.05 PR-AUC and 0.05 event recall at the 1-per-1,000 false-alarm budget, and false alarms are no worse. The 0.05 is my choice, not a measured noise level. If the rule isn't met, the honest wording is "no gain on this encoder". It doesn't show that long memory is useless, because the LSTM only sees what the frozen encoder passes on.

Stage 2 only sees 3 years of weather, so a long run can overfit. After each epoch the log prints the loss on a validation year next to the training loss. The best epoch by validation loss is saved as `<job>_s42_best.pt`. The final model (last epoch) is the one that gets scored.

## Confidence intervals

The last cell runs `python -m qc paired`. It gives the difference between each job and the stage-1 run for every fault kind, with a paired resample of the 5 validation years (all 3,125 combinations, regions fixed). I resample years and not events because events in one year share the weather, which makes event-level intervals far too narrow (see `docs/research/bootstrap-unit-check.md`). Five years is still few, so the intervals undercover and are only a rough guide. The output also lists the difference per year and how many years each side wins.

Everything is on synthetic faults and validation years, with one seed.

## How to run

1. New Notebook, File > Import Notebook, upload this file.
2. Accelerator: GPU (T4 x2 or a single GPU). Internet: On.
3. Add Input: the RF25 dataset `nathashalewis/majorproj-rainfall`, and a private dataset or Kaggle model (framework PyTorch) that holds `stnet_s42_final.pt` (or `stnet_hold_s42_final.pt` for the second session). Don't upload it as a .zip, Kaggle would unpack it. Copy its mounted path from the Input panel into `RESUME_LINK` in the next cell.
4. Save Version > Save & Run All (Commit), so it runs in the background and the output is kept.
5. Download `stnet_results_<jobs>.zip` from the Output tab. It has the reports, the paired comparison, the logs and the weights (`_final`, `_best`, `_last`). The weights are also in `artifacts/qc/base/<job>_s42/checkpoints/`.

Each job's scoring runs as soon as that job finishes. The paired comparison runs last.

If a job hits the time cap or the session dies, save the version, attach its output as an input, put that path in `RESUME_LINK` and run again with the same job settings. A resume refuses a changed run length.

In [ ]:
RUN_JOBS = ['lmb', 'lmc']
SMOKE = False
DATASET_LINK = '/kaggle/input/datasets/nathashalewis/majorproj-rainfall'
REPO_URL = 'https://github.com/NL0711/datamend.git'
BRANCH = 'main'
RUN_NAME = 'base'
EPOCHS, STEPS, LONG_STEPS = (4, 200, 30)
TIME_CAP_HOURS = 8.0
RESUME_LINK = ''
EXPECTED_BENCH_HASH = 'c37de7e3cf2be90e'
HOLDOUT = ['--holdout-kinds', 'zeroed', 'scale', 'drift']
ENCODERS = {
    'stnet':      dict(prefix='stnet',      seed=42, args=[]),
    'stnet_hold': dict(prefix='stnet_hold', seed=42, args=HOLDOUT),
}
JOBS = {
    'lmb':  dict(prefix='stnet_lmb',  seed=42, encoder='stnet',      args=['--init-from', 'stnet', '--long-epochs', '8',  '--long-steps', '300'], post=True),
    'lmc':  dict(prefix='stnet_lmc',  seed=42, encoder='stnet',      args=['--init-from', 'stnet', '--long-epochs', '30', '--long-steps', '600'], post=True),
    'hlmb': dict(prefix='stnet_hlmb', seed=42, encoder='stnet_hold', args=['--init-from', 'stnet_hold', *HOLDOUT, '--long-epochs', '8',  '--long-steps', '300'], post=True),
    'hlmc': dict(prefix='stnet_hlmc', seed=42, encoder='stnet_hold', args=['--init-from', 'stnet_hold', *HOLDOUT, '--long-epochs', '30', '--long-steps', '600'], post=True),
}
unknown = [j for j in RUN_JOBS if j not in JOBS]
if unknown: raise ValueError('unknown job(s): ' + ', '.join(unknown) + '. Choose from ' + ', '.join(JOBS))
if len({JOBS[j]['encoder'] for j in RUN_JOBS}) > 1: raise ValueError('one session uses one encoder: run [lmb, lmc] or [hlmb, hlmc], not a mix')
print('jobs:', RUN_JOBS, '| encoder:', JOBS[RUN_JOBS[0]]['encoder'], '| run name:', RUN_NAME)

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time, importlib
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').is_dir() else pathlib.Path.cwd()
def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if show: print('$', cmd, chr(10) + out[-1200:])
    if r.returncode: raise RuntimeError('failed: ' + cmd[:120] + chr(10) + out[-1200:])
    return out
os.environ['GIT_TERMINAL_PROMPT'] = '0'
REPO = WORK / 'datamend'
try:
    if (REPO / '.git').is_dir():
        sh('git -C ' + str(REPO) + ' fetch ' + REPO_URL + ' ' + BRANCH + ' && git -C ' + str(REPO) + ' checkout -B ' + BRANCH + ' FETCH_HEAD')
    else:
        sh('git clone --branch ' + BRANCH + ' ' + REPO_URL + ' ' + str(REPO))
except RuntimeError as e:
    raise RuntimeError('Could not get the repository. Check that Internet is ON in the notebook settings.') from None
sh('git -C ' + str(REPO) + ' log -1 --oneline')
ML = REPO / 'ml-service'
os.chdir(ML)
sys.path.insert(0, str(ML))
for _m in [k for k in sys.modules if k == 'qc' or k.startswith('qc.')]:
    del sys.modules[_m]
importlib.invalidate_caches()
sh('pip install -q numpy pandas scipy scikit-learn xarray netCDF4 matplotlib pyyaml pyarrow kagglehub psutil nvidia-ml-py', show=False)
print('repo:', REPO, '| cwd:', os.getcwd())

In [ ]:
import torch
from qc.kaggle_data import locate_rf25, find_saved_outputs
from qc.config import load_config, year_range

cfg0 = load_config(ML / 'qc' / 'configs' / 'base.yaml', [])
need = sum(len(year_range(cfg0.years[k])) for k in ('train', 'val', 'test', 'stress') if cfg0.years.get(k))
def _download(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)
RF25_DIR = locate_rf25(DATASET_LINK, expected_files=need, download=_download)
n_gpu = torch.cuda.device_count()
print('RF25:', RF25_DIR, '|', len(list(RF25_DIR.glob('RF25_ind*_rfp25.nc'))), 'files | GPUs:', n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
if n_gpu == 0: raise RuntimeError('No GPU. This code needs a CUDA GPU (T4 or P100). A TPU notebook cannot run it: change the accelerator in the notebook settings.')
if len(RUN_JOBS) > n_gpu: print('Only', n_gpu, 'GPU(s) for', len(RUN_JOBS), 'jobs: they will run one after the other.')
os.environ['DATAMEND_RF25_DIR'] = str(RF25_DIR)
os.environ['DATAMEND_DATA_DIR'] = str(WORK / 'data')
OUT_ROOT = WORK / 'artifacts'
RUN = RUN_NAME
BASE = ['--profile', 'single-gpu', '--set', 'name=' + RUN]

if not RESUME_LINK.strip():
    raise RuntimeError('RESUME_LINK is empty. Attach the dataset or model that holds the weights file and put its mounted path (Input panel) in RESUME_LINK in the first cell.')
root = pathlib.Path(RESUME_LINK.strip())
if not root.is_dir():
    raise RuntimeError('RESUME_LINK is not a folder here: ' + str(root) + '. Copy the path from the Input panel.')
prev = find_saved_outputs(RESUME_LINK)
if prev is not None:
    for sub in ('artifacts', 'data'):
        if (prev / sub).is_dir():
            shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
    print('restored a previous output from', prev)
import zipfile
ENCODER_USED = ENCODERS[JOBS[RUN_JOBS[0]]['encoder']]
for _seed in sorted({JOBS[n]['seed'] for n in RUN_JOBS}):
    stem = ENCODER_USED['prefix'] + '_s' + str(_seed)
    nm = stem + '_final'
    ck = WORK / 'artifacts' / 'qc' / RUN / stem / 'checkpoints' / (nm + '.pt')
    ck.parent.mkdir(parents=True, exist_ok=True)
    if not ck.is_file():
        files = [f for f in sorted(root.rglob(nm + '.pt')) if f.is_file()]
        if files:
            shutil.copyfile(files[0], ck)
            print('weights file taken from', files[0])
        else:
            folders = [d for d in sorted(root.rglob(nm)) if d.is_dir() and (d / 'data.pkl').is_file()]
            if folders:
                with zipfile.ZipFile(ck, 'w', zipfile.ZIP_STORED) as z:
                    for f in sorted(folders[0].rglob('*')):
                        if f.is_file():
                            z.write(f, pathlib.PurePosixPath(nm) / f.relative_to(folders[0]).as_posix())
                print('weights file rebuilt from the unzipped folder', folders[0])
    if not ck.is_file():
        raise RuntimeError(nm + '.pt was not found under ' + str(root) + '. Upload the file as a private dataset or Kaggle model (not as a .zip), or attach the output of the run that made it. If it is lost, that run (about 2.5 hours) has to be repeated first.')
    print('stage-1 checkpoint ready:', ck.stat().st_size // 1024, 'KB at', ck)

In [ ]:
def run_stream(step, *args):
    cmd = [sys.executable, '-u', '-m', 'qc', step, *args, *BASE]
    print('$', ' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ML)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('step failed: ' + step)

run_stream('prepare')
run_stream('benchmark')
run_stream('untrained')

from qc.benchmark import bench_dir
cfg = load_config(ML / 'qc' / 'configs' / 'base.yaml', ['name=' + RUN])
bench_hash = json.loads((bench_dir(cfg) / 'manifest.json').read_text())['hash']
print('benchmark hash:', bench_hash)
if bench_hash != EXPECTED_BENCH_HASH:
    raise RuntimeError('benchmark hash ' + bench_hash + ' is not the expected ' + EXPECTED_BENCH_HASH + ': results from this notebook would not be comparable with the earlier ones. Stop and check the repository version.')

In [ ]:
RUN_ERRORS = []
unfinished = []
def capped(name):
    try:
        return 'stopped at the time cap' in str((WORK / ('job_' + name + '.log')).read_text().splitlines()[-3:])
    except Exception:
        return False
def run_post(name):
    j = JOBS[name]
    s, pre = str(j['seed']), j['prefix']
    run_stream('ablate', '--seed', s, '--tag', pre, '--subset', f'rules+{pre}_fault', '--subset', f'rules+{pre}_fault+{pre}_clean',
               '--subset', f'rules+{pre}_fault+{pre}_zeroed+{pre}_scale+{pre}_drift')
    run_stream('ceiling', '--seed', s, '--tag', pre, '--components', *[f'{pre}_{k}_s{s}' for k in ('fault', 'clean', 'disc', 'zeroed', 'scale', 'drift')])
    run_stream('calibrate', '--seed', s, '--components', f'{pre}_fault_s{s}')

queue = list(RUN_JOBS)
free = list(range(n_gpu))
running = {}
done = {}
def launch(name, gpu):
    j = JOBS[name]
    cmd = [sys.executable, '-u', '-m', 'qc', 'stnet', '--seeds', str(j['seed']), '--prefix', j['prefix'], '--epochs', str(EPOCHS), '--steps', str(STEPS),
           '--long-steps', str(LONG_STEPS), '--time-cap-hours', str(TIME_CAP_HOURS), *j['args'], *BASE, '--set', 'compute.devices=[cuda:' + str(gpu) + ']']
    logf = open(WORK / ('job_' + name + '.log'), 'a')
    running[name] = (subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, text=True, cwd=ML), logf, gpu)
    print('started', name, 'on cuda:' + str(gpu), '|', ' '.join(cmd[4:]))

while queue or running:
    while queue and free:
        launch(queue.pop(0), free.pop(0))
    time.sleep(60)
    ended = []
    for name in list(running):
        p, logf, gpu = running[name]
        rc = p.poll()
        try:
            tail = (WORK / ('job_' + name + '.log')).read_text().strip().splitlines()[-1][:150]
        except Exception:
            tail = ''
        print(time.strftime('%H:%M'), name, 'running' if rc is None else 'finished rc=' + str(rc), '|', tail)
        if rc is not None:
            logf.close()
            del running[name]
            free.append(gpu)
            done[name] = rc
            ended.append(name)
    while queue and free:
        launch(queue.pop(0), free.pop(0))
    for name in ended:
        if done[name]:
            RUN_ERRORS.append(name + ' failed (rc=' + str(done[name]) + '), see /kaggle/working/job_' + name + '.log')
        elif capped(name):
            unfinished.append(name)
            print('Time cap reached for', name, '- save the version, attach the output as an input, set RESUME_LINK and run again with the same job settings. Scoring is skipped for it.')
        elif JOBS[name]['post']:
            try:
                run_post(name)
            except Exception as e:
                RUN_ERRORS.append(name + ' scoring failed: ' + str(e)[:300])
                print('scoring failed for', name, '(the other job and the zip are not affected):', str(e)[:300])
print('jobs ended:', done, '| time-capped:', unfinished, '| errors:', RUN_ERRORS)

In [ ]:
ok_jobs = [n for n in RUN_JOBS if done.get(n) == 0 and n not in unfinished]
if ok_jobs:
    try:
        e = ENCODER_USED
        run_stream('stnet', '--seeds', str(e['seed']), '--prefix', e['prefix'], '--epochs', str(EPOCHS), '--steps', str(STEPS), '--long-steps', str(LONG_STEPS),
                   *e['args'], '--set', 'compute.devices=[cuda:0]')
        run_stream('paired', '--base', e['prefix'], '--others', *[JOBS[n]['prefix'] for n in ok_jobs], '--seed', str(e['seed']), '--budget', '1.0')
    except Exception as ex:
        RUN_ERRORS.append('paired comparison failed: ' + str(ex)[:300])
        print('paired comparison failed (the rest is unaffected):', str(ex)[:300])
else:
    print('no job finished cleanly, so no paired comparison')
for name in RUN_JOBS:
    print(name, '| trained:', done.get(name) == 0, '| time-capped:', name in unfinished)

In [ ]:
import zipfile
run_dir = OUT_ROOT / 'qc' / RUN
prefixes = [JOBS[n]['prefix'] for n in RUN_JOBS]
keep = []
for pat in ('**/*.md', '**/*.json', '**/*.png'):
    keep += [p for p in run_dir.glob(pat) if p.is_file() and p.stat().st_size < 50e6 and 'scores' not in p.parts and 'checkpoints' not in p.parts]
weights = [p for p in run_dir.glob('**/checkpoints/*.pt') if p.is_file() and any(x + '_s' in p.name for x in prefixes)]
logs = [WORK / ('job_' + n + '.log') for n in RUN_JOBS if (WORK / ('job_' + n + '.log')).is_file()]
keep = list(dict.fromkeys(keep + weights + logs))
zname = WORK / ('stnet_results_' + '_'.join(RUN_JOBS) + '.zip')
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, p.relative_to(WORK))
print(zname, '(' + format(zname.stat().st_size / 1e6, '.1f') + ' MB,', len(keep), 'files). Weights in it:', sorted(w.name for w in weights))
if not any(w.name.endswith('_final.pt') for w in weights):
    RUN_ERRORS.append('no _final.pt weights were written, no job finished')
print('Download', zname.name, 'from the Output tab.')
if RUN_ERRORS:
    raise RuntimeError('; '.join(RUN_ERRORS))